In [1]:
## Use gpt-oss-20b to solve text based problems.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA A100-SXM4-80GB


In [3]:
model_name="gpt-oss-20b"
model_id = "openai/gpt-oss-20b"

# Download and load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Download the model weights and load onto available devices
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
)

config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/36.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [4]:
## import the prompts from the text_prompts.py
import sys

sys.path.append("/content/drive/MyDrive/colab_files/Problems/Text")
from text_prompts import prompts
print(prompts)

{'regular_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem. The question has equations and symbols in latex, but your answer should not be in latex.', 'detail_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem, but if there are any physical laws broken then mention it and also say if the problem can be solved or not. The question has equations and symbols in latex, but your answer should not be in latex.'}


In [5]:
print(prompts.keys())

dict_keys(['regular_system_prompt', 'detail_system_prompt'])


In [6]:
from pathlib import Path

text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text")
file_names = sorted(file.name for file in text_folder.glob("*.md"))
print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 9
['Problem_1_Obvious_Q.md', 'Problem_1_Regular_Q.md', 'Problem_1_non_obvious_Q.md', 'Problem_2_Obvious_Q.md', 'Problem_2_Regular_Q.md', 'Problem_2_non_obvious_Q.md', 'Problem_3_Obvious_Q.md', 'Problem_3_Regular_Q.md', 'Problem_3_non_obvious_Q.md']


In [7]:
%%time

for keys in prompts:
  for file in file_names:
    text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text")
    file_path = text_folder / file

    if keys == "detail_system_prompt" and "regular" in file_path.stem.lower():  ## skip if keys have detail_system_prompt and
                continue     ## problems that have regular in their name. We dont need Regular problems to use detail prompts but run the rest

    problem_text = file_path.read_text(encoding="utf-8")

    messages = [
    {"role": "system", "content":prompts[keys]},

     {"role":"user", "content": problem_text  }
        ]

    inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
                  ).to(model.device)

    outputs = model.generate(
        **inputs, max_new_tokens=10000)

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)


    ## store the output
    output_filename = f"{model_name}_{file_path.stem}_{keys}.md"
    output_folder = Path("/content/drive/MyDrive/colab_files/Results/gpt_oss_20b")

    output_file_path = output_folder / output_filename
    output_file_path.write_text(output_text, encoding="utf-8")


    ## Get metadata

    # Remove the input portion to obtain only generated tokens
    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0, input_length:]

    # Count tokens for this single conversation
    input_tokens = int(inputs["attention_mask"][0].sum().item())
    output_tokens = int(generated_tokens.numel())

    metadata = {
      "requested_model": model_name,
      "problem_text":problem_text ,
      "prompt_name":keys,
      "system_prompt":prompts[keys],

      "token_usage": {
          "input_tokens": input_tokens,
          "output_tokens": output_tokens,
          "total_tokens": input_tokens + output_tokens,
            }
            }

    ## store the metadata
    metadata_output_filename = f"{model_name}_{file_path.stem}_{keys}.json"

    output_folder = Path("/content/drive/MyDrive/colab_files/Results/metadata")

    metadata_output_file_path = output_folder / metadata_output_filename
    metadata_output_file_path.write_text(
                        json.dumps(metadata, indent=4, ensure_ascii=False),
                        encoding="utf-8")
    print(f"Completed:::{model_name}_{file_path.stem}_{keys}")

Completed:::gpt-oss-20b_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_1_Regular_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_1_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_2_Regular_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_3_Regular_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_3_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-20b_Problem_1_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-20b_Problem_1_non_obvious_Q_detail_system_prompt
Completed:::gpt-oss-20b_Problem_2_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-20b_Problem_2_non_obvious_Q_detail_system_prompt
Completed:::gpt-oss-20b_Problem_3_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-20b_Problem_3_non_obvious_Q_detail_system